# sistemas recomendadores de peliculas

en este notebook se construyen un recomendador simple uno basado en contenido y una version mejorada con metadatos

In [ ]:
import ast
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce').fillna(0)
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce').fillna(0)
df.head(3)

## recomendador simple con media ponderada

la formula combina la calificacion promedio con la cantidad de votos para evitar que una pelicula con pocos votos quede artificialmente arriba

In [ ]:
C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.90)
q_movies = df.loc[df['vote_count'] >= m].copy()

def calificacion_ponderada(fila, m=m, C=C):
    v = fila['vote_count']
    R = fila['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

q_movies['score'] = q_movies.apply(calificacion_ponderada, axis=1)
top_15 = q_movies.sort_values('score', ascending=False)
top_15[['title', 'vote_count', 'vote_average', 'score']].head(15)

## recomendador basado en contenido

se usa la sinopsis de cada pelicula como texto y se calcula la similitud coseno entre sus vectores tf idf

In [ ]:
df['overview'] = df['overview'].fillna('')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recomendar_por_sinopsis(titulo, matriz=cosine_sim):
    indice = indices[titulo]
    puntajes = list(enumerate(matriz[indice]))
    puntajes = sorted(puntajes, key=lambda x: x[1], reverse=True)[1:11]
    indices_peliculas = [indice for indice, _ in puntajes]
    return df['title'].iloc[indices_peliculas].tolist()

recomendar_por_sinopsis('The Dark Knight Rises')

## recomendador mejorado con metadatos

se agregan actores director generos palabras clave guionistas y productores. el director aparece tres veces para darle mas peso

In [ ]:
credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')
df_meta = df.drop([19730, 29503, 35587, 35803], errors='ignore').copy()
for tabla in (credits, keywords, df_meta):
    tabla['id'] = pd.to_numeric(tabla['id'], errors='coerce')
df_meta = df_meta.dropna(subset=['id']).copy()
credits = credits.dropna(subset=['id']).copy()
keywords = keywords.dropna(subset=['id']).copy()
df_meta['id'] = df_meta['id'].astype(int)
credits['id'] = credits['id'].astype(int)
keywords['id'] = keywords['id'].astype(int)
df_meta = df_meta.merge(credits, on='id').merge(keywords, on='id')

for columna in ['cast', 'crew', 'keywords', 'genres']:
    df_meta[columna] = df_meta[columna].apply(ast.literal_eval)

def obtener_director(equipo):
    for persona in equipo:
        if persona.get('job') == 'Director':
            return persona.get('name', '')
    return ''

def obtener_lista(elementos, limite=3):
    return [elemento.get('name', '') for elemento in elementos[:limite]] if isinstance(elementos, list) else []

def obtener_equipo(equipo, trabajos):
    return [persona.get('name', '') for persona in equipo if persona.get('job') in trabajos][:3]

df_meta['director'] = df_meta['crew'].apply(obtener_director)
df_meta['cast'] = df_meta['cast'].apply(obtener_lista)
df_meta['keywords'] = df_meta['keywords'].apply(obtener_lista)
df_meta['genres'] = df_meta['genres'].apply(obtener_lista)
df_meta['writers'] = df_meta['crew'].apply(lambda x: obtener_equipo(x, {'Writer', 'Screenplay'}))
df_meta['producers'] = df_meta['crew'].apply(lambda x: obtener_equipo(x, {'Producer', 'Executive Producer'}))

columnas = ['cast', 'keywords', 'genres', 'writers', 'producers']
for columna in columnas:
    df_meta[columna] = df_meta[columna].apply(lambda valores: [str(valor).lower().replace(' ', '') for valor in valores])
df_meta['director'] = df_meta['director'].fillna('').str.lower().str.replace(' ', '', regex=False)

def crear_sopa(fila):
    director = ' '.join([fila['director']] * 3)
    return ' '.join(fila['keywords'] + fila['cast'] + fila['genres'] + fila['writers'] + fila['producers'] + [director])

df_meta['soup'] = df_meta.apply(crear_sopa, axis=1)
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df_meta['soup'])
cosine_sim_meta = cosine_similarity(count_matrix, count_matrix)
df_meta = df_meta.reset_index(drop=True)
indices_meta = pd.Series(df_meta.index, index=df_meta['title']).drop_duplicates()

def recomendar_con_popularidad(titulo, matriz=cosine_sim_meta, cantidad_similares=30, cantidad_final=10):
    indice = indices_meta[titulo]
    puntajes = sorted(enumerate(matriz[indice]), key=lambda x: x[1], reverse=True)[1:cantidad_similares + 1]
    candidatos = df_meta.iloc[[indice for indice, _ in puntajes]].copy()
    candidatos['score'] = candidatos.apply(calificacion_ponderada, axis=1)
    return candidatos.sort_values('score', ascending=False)[['title', 'vote_average', 'vote_count', 'score']].head(cantidad_final)

recomendar_con_popularidad('The Dark Knight Rises')